# BLR (KIA) Flight KPI Analysis

Five core KPIs over **24h**, **7d**, and **30d** windows from PostgreSQL `flights`.

| KPI | SQL file |
|-----|----------|
| On-time departure % | `sql/01_on_time_departure_rate.sql` |
| On-time arrival % | `sql/02_on_time_arrival_rate.sql` |
| Avg departure delay | `sql/03_avg_departure_delay.sql` |
| Avg arrival delay | `sql/04_avg_arrival_delay.sql` |
| Airline ranking | `sql/05_airline_ranking.sql` |
| **All metrics** | `sql/99_kpi_metrics_unified.sql` |

Output columns: `airline_iata`, `metric_name`, `metric_value`, `window_type`, `calculated_at`

In [ ]:
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(Path('..') / '.env')

PROJECT_ROOT = Path('..').resolve()
SQL_DIR = PROJECT_ROOT / 'sql'

CONN = dict(
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
)


def run_sql_file(path: Path) -> None:
    sql = path.read_text()
    with psycopg2.connect(**CONN) as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql)
    print(f'Executed: {path.name}')


def query_df(sql: str) -> pd.DataFrame:
    with psycopg2.connect(**CONN) as conn:
        return pd.read_sql_query(sql, conn)


def query_file(name: str) -> pd.DataFrame:
    return query_df((SQL_DIR / name).read_text())

In [ ]:
# Bootstrap views (idempotent)
run_sql_file(SQL_DIR / '00_flight_kpi_base_view.sql')
run_sql_file(SQL_DIR / '99_kpi_metrics_unified.sql')
run_sql_file(SQL_DIR / 'refresh_kpi_metrics.sql')

## Unified KPI output (all 5 KPIs)

In [ ]:
all_kpis = query_df('SELECT * FROM v_kpi_metrics_unified ORDER BY window_type, metric_name, airline_iata')
all_kpis.head(20)

In [ ]:
pivot = all_kpis.pivot_table(
    index=['airline_iata', 'window_type'],
    columns='metric_name',
    values='metric_value',
    aggfunc='first',
)
pivot.filter(like='on_time').head(15)

## KPI 1 — On-time departure %

In [ ]:
query_file('01_on_time_departure_rate.sql').head(20)

## KPI 2 — On-time arrival % (primary)

In [ ]:
query_file('02_on_time_arrival_rate.sql').head(20)

## KPI 3 & 4 — Average delays

In [ ]:
display(query_file('03_avg_departure_delay.sql').head(10))
display(query_file('04_avg_arrival_delay.sql').head(10))

## KPI 5 — Airline composite ranking (min 5 flights)

In [ ]:
ranking = query_file('05_airline_ranking.sql')
ranking[ranking['metric_name'] == 'airline_rank'].sort_values(['window_type', 'metric_value'])

## Persist snapshot (optional)

In [ ]:
inserted = query_df('SELECT refresh_kpi_metrics() AS rows_inserted')
inserted